[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/05_00_main_intuition.ipynb)

# NLP intuition: two small demos

Two ideas:

1. **Tokenization is not one thing.** Different models split the same sentence differently. You can *see* it.
2. **Bag-of-words throws away order.** Two sentences with opposite meanings can end up with identical vectors.

That's it. Run the cells, look at the printouts, build the mental model.

## Setup

We need HuggingFace `transformers` for the tokenizers. On Colab uncomment the pip line.

In [ ]:
# !pip install -q transformers

from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Part 1: Same sentence, different splits

A *tokenizer* turns a string into a list of integers a model can consume. There are several common strategies:

- **GPT-2** uses byte-pair encoding (BPE). Whitespace is part of the token.
- **BERT** uses WordPiece. Sub-pieces of a word are marked with `##`.

Same sentence, two tokenizers. Watch how the splits differ.

In [ ]:
tokenizers = {
    "gpt2": AutoTokenizer.from_pretrained("gpt2"),
    "bert": AutoTokenizer.from_pretrained("bert-base-uncased"),
}

sentence = "Tokenization is the unsexy foundation of modern NLP."

for name, tok in tokenizers.items():
    pieces = tok.tokenize(sentence)
    print(f"{name:5} ({len(pieces):2} tokens): {pieces}")

### What about words the model has never seen?

Subword tokenizers don't have an "unknown" failure mode for most inputs, they just keep splitting until the pieces are in the vocabulary. The pieces themselves are informative: long made-up words shatter into many tokens, and a model with a primarily-English vocab spends more tokens per word on other languages.

This matters practically: **API costs scale with token count**, not character count.

In [ ]:
weird = [
    "antidisestablishmentarianism",
    "🎉 emoji party 🎉",
    "Le chat noir dort sur la table.",
    "def foo(x): return x ** 2",
]

for s in weird:
    print(f"\nINPUT: {s}")
    for name, tok in tokenizers.items():
        pieces = tok.tokenize(s)
        print(f"  {name:5} ({len(pieces):2}): {pieces}")

## Part 2: Bag-of-words throws away order

TF–IDF (and any plain bag-of-words representation) counts which words appear, not what order they appear in. That has a sharp consequence: two sentences with the *same words in different order* are indistinguishable.

Classic example:

- "dog bites man": not news
- "man bites dog": news

Different meanings. Same vector.

In [ ]:
docs = ["dog bites man", "man bites dog"]

vec = TfidfVectorizer()
X = vec.fit_transform(docs)

print("Vocabulary:", vec.get_feature_names_out())
print("\nVectors:")
print(X.toarray())
print(f"\nCosine similarity: {cosine_similarity(X[0], X[1])[0, 0]:.3f}")

### A partial fix: n-grams

Adding bigrams (pairs of adjacent words) preserves a little local order. Now `"dog bites"` and `"bites man"` are different features from `"man bites"` and `"bites dog"`, so the two docs no longer collide.

This is partial. N-grams blow up vocabulary size and still miss long-range structure. The real fix, attention, is what transformers give you.

In [ ]:
vec_bi = TfidfVectorizer(ngram_range=(1, 2))
X_bi = vec_bi.fit_transform(docs)

print("Vocabulary:", vec_bi.get_feature_names_out())
print(f"\nCosine similarity (with bigrams): {cosine_similarity(X_bi[0], X_bi[1])[0, 0]:.3f}")

## Takeaways

1. **Tokenization is a choice**: Different models see different units. The same string is a different number of tokens depending on who's asking.
2. **Subword tokenizers don't choke on unfamiliar input**... they just spend more tokens on it. 
3. **Bag-of-words representations are order-blind by construction.** If word order matters for your task, TF–IDF is the wrong tool. N-grams help a little; attention helps a lot.